<a href="https://colab.research.google.com/github/arac123/data/blob/main/%EC%98%A4%ED%94%88%EC%86%8C%EC%8A%A4_%EB%8D%B0%EC%9D%B4%ED%84%B0_%EB%B6%84%EC%84%9D_4%EA%B0%95.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



# 오픈소스 기반 데이터 분석 4강 - 데이터 수집


## 4-1 CSV 파일 읽기

In [6]:
import pandas as pd

## data.csv 파일 읽기
df = pd.read_csv('data.csv', encoding='utf-8', sep=',', header=0,
                 index_col=None, skiprows=None, nrows=None)

print(df)

           날짜    체중  골격근량  체지방량
0  2025.02.06  64.7  30.0  11.1
1  2025.02.04  64.0  29.3  11.6


## 4-2 JSON 파일 읽기



In [12]:

import json
import pandas as pd

## data.json 파일 출력
with open('data.json', mode='r', encoding='utf-8') as f :
        data = json. load(f)
print(data)

## data.json 파일 DataFrame 읽기
df = pd.read_json('data.json', orient='records', encoding='utf-8')
print(df)

{'매출데이터': [{'월': '2025-01', '매출액': 1000000, '비용': 700000, '이익': 300000}, {'월': '2025-02', '매출액': 1200000, '비용': 800000, '이익': 400000}, {'월': '2025-03', '매출액': 1500000, '비용': 900000, '이익': 600000}]}
                                               매출데이터
0  {'월': '2025-01', '매출액': 1000000, '비용': 700000,...
1  {'월': '2025-02', '매출액': 1200000, '비용': 800000,...
2  {'월': '2025-03', '매출액': 1500000, '비용': 900000,...


## 4-3 텍스트 파일 읽기 및 데이터 추출

In [ ]:
import re

## 파일(callcenter20250301.log) 오픈 및 읽기
with open('callcenter20250301.log', 'r', encoding='utf-8') as f:
    content = f.read()
## 주민등록번호 패턴 생성

## 주민등록번호 마스킹


## 마스킹된 파일(callcenter20250301_masked.log) 오픈 및 쓰기


print("주민등록번호 마스킹 완료. 'callcenter20250301_masked.log.txt' 파일로 저장되었습니다.")

## 4-4 Open-Meteo의 무료 날씨 API를 통한 특정 지역 온도 조회

In [13]:
import pandas as pd
import requests

# 1. 설정 및 파라미터 정의
SERVICE_KEY = ""  # 공공데이터포털 인증키 입력 (비워둠)
BASE_URL = "https://apis.data.go.kr/B552845/perDay/price"

YEARS = [str(year) for year in range(2015, 2025)]  # 2015 ~ 2024

# 배추 품종 코드 정의 (API 명세 기준)
# 01: 봄배추, 02: 여름(고랭지)배추, 03: 가을배추, 04: 월동배추
VARIETIES = {
    "01": "봄배추",
    "02": "여름(고랭지)배추",
    "03": "가을배추",
    "04": "월동배추",
}

all_data = []

# 2. 연도 및 품종별 데이터 수집 Loop
for year in YEARS:
    for vrty_cd, vrty_nm in VARIETIES.items():
        params = {
            "serviceKey": SERVICE_KEY,
            "pageNo": "1",
            "numOfRows": "1000",  # 연간 해당 품종 데이터 충분히 수집
            "returnType": "JSON",
            "cond[exmn_ymd::GTE]": f"{year}0101",
            "cond[exmn_ymd::LTE]": f"{year}1231",
            "cond[se_cd::EQ]": "02",  # 도매
            "cond[ctgry_cd::EQ]": "200",  # 채소류
            "cond[item_cd::EQ]": "211",  # 배추
            "cond[grd_cd::EQ]": "04",  # 상품
            "cond[sgg_cd::EQ]": "1101",  # 서울
            "cond[mrkt_cd::EQ]": "0110211",  # 가락도매
            "cond[vrty_cd::EQ]": vrty_cd,  # 품종코드
        }

        try:
            response = requests.get(BASE_URL, params=params, timeout=10)
            if response.status_code == 200:
                res_json = response.json()
                # API 응답 구조 확인 및 데이터 추출
                items = (
                    res_json.get("response", {})
                    .get("body", {})
                    .get("items", {})
                    .get("item", [])
                )

                if isinstance(items, dict):  # 단일 데이터일 경우 리스트화
                    items = [items]

                for item in items:
                    item["vrty_nm"] = vrty_nm  # 품종명 추가
                    all_data.append(item)

                print(
                    f"[{year}년 / {vrty_nm}] 수집 완료 - {len(items)}건"
                )
            else:
                print(
                    f"[{year}년 / {vrty_nm}] 호출 실패 (Status: {response.status_code})"
                )
        except Exception as e:
            print(f"[{year}년 / {vrty_nm}] 에러 발생: {e}")

# 3. 데이터프레임 변환 및 저장
df = pd.DataFrame(all_data)

# 주요 컬럼 추출 및 정렬 (조사일자, 품종명, 가격 등)
if not df.empty:
    columns_to_keep = [
        "exmn_ymd",
        "vrty_nm",
        "item_nm",
        "mrkt_nm",
        "avrg_prc",
    ]
    # 존재하는 컬럼만 선택
    available_cols = [c for c in columns_to_keep if c in df.columns]
    df_result = df[available_cols]

    # Excel 저장
    df_result.to_excel(
        "가락시장_배추_가격데이터_2015_2024.xlsx", index=False
    )
    print("\n데이터 수집 및 엑셀 저장 완료!")

[2015년 / 봄배추] 호출 실패 (Status: 401)
[2015년 / 여름(고랭지)배추] 호출 실패 (Status: 401)
[2015년 / 가을배추] 호출 실패 (Status: 401)
[2015년 / 월동배추] 호출 실패 (Status: 401)
[2016년 / 봄배추] 호출 실패 (Status: 401)
[2016년 / 여름(고랭지)배추] 호출 실패 (Status: 401)
[2016년 / 가을배추] 호출 실패 (Status: 401)
[2016년 / 월동배추] 호출 실패 (Status: 401)
[2017년 / 봄배추] 호출 실패 (Status: 401)
[2017년 / 여름(고랭지)배추] 호출 실패 (Status: 401)
[2017년 / 가을배추] 호출 실패 (Status: 401)
[2017년 / 월동배추] 호출 실패 (Status: 401)
[2018년 / 봄배추] 호출 실패 (Status: 401)
[2018년 / 여름(고랭지)배추] 호출 실패 (Status: 401)
[2018년 / 가을배추] 호출 실패 (Status: 401)
[2018년 / 월동배추] 호출 실패 (Status: 401)
[2019년 / 봄배추] 호출 실패 (Status: 401)
[2019년 / 여름(고랭지)배추] 호출 실패 (Status: 401)
[2019년 / 가을배추] 호출 실패 (Status: 401)
[2019년 / 월동배추] 호출 실패 (Status: 401)
[2020년 / 봄배추] 호출 실패 (Status: 401)
[2020년 / 여름(고랭지)배추] 호출 실패 (Status: 401)
[2020년 / 가을배추] 호출 실패 (Status: 401)
[2020년 / 월동배추] 호출 실패 (Status: 401)
[2021년 / 봄배추] 호출 실패 (Status: 401)
[2021년 / 여름(고랭지)배추] 호출 실패 (Status: 401)
[2021년 / 가을배추] 호출 실패 (Status: 401)
[2021년 / 월동배추] 호출 실패 (Statu

## 4-5 Selenium과 lxml을 이용한 웹 스크래핑

In [ ]:
!curl -o google-chrome-stable_current_amd64.deb https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt install ./google-chrome-stable_current_amd64.deb -y
!pip install selenium webdriver_manager

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service as ChromeService
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from lxml import html
import time

chrome_options = webdriver.ChromeOptions()
chrome_options.add_argument('--headless')               # 브라우저 창 없이 실행
chrome_options.add_argument('--no-sandbox')             # 보안모드 비활성화 (Colab 필수)
chrome_options.add_argument('--disable-dev-shm-usage')  # 메모리 부족 방지 (Colab 필수)
chrome_options.add_argument('--window-size=1920x1080')  # 창 크기 설정(가상)
chrome_options.add_argument('--disable-gpu')            # GPU 가속 비활성화 (일부 환경 안정성)
chrome_options.binary_location = "/usr/bin/google-chrome-stable"  # Colab용 크롬 경로 지정

## 드라이버 실행
driver = webdriver.Chrome(options=chrome_options)

## 사이트 접속


## 사이트 접속 대기


## 페이지 제목 출력


## 드라이버 종료
driver.quit()


# 실습 시나리오

## 공공데이터 포털 가입 및 데이터 신청

- [https://www.data.go.kr](https://www.data.go.kr)
- 한국환경공단 에어코리아 대기오염정보 데이터 신청

In [14]:
import requests

## 데이터 수집 url 및 api key 설정
https://apis.data.go.kr/B552845/perDay/price?serviceKey={7adbc84be8cddee524a52af7e66c8cdbbefea83604ad4aeee8aafd6ba6b9d947}&pageNo=1&numOfR
ows=200&returnType=JSON&cond[exmn_ymd::GTE]={2015}0101&cond[exmn_ymd::LTE]={2024}1
231&cond[se_cd::EQ]=02&cond[ctgry_cd::EQ]=200&cond[item_cd::EQ]=211&cond[grd_cd::EQ]=0
4&cond[sgg_cd::EQ]=1101&cond[mrkt_cd::EQ]=0110211&cond[vrty_cd::EQ]={01 / 02 / 03 / 06}

params = {
    'serviceKey':7adbc84be8cddee524a52af7e66c8cdbbefea83604ad4aeee8aafd6ba6b9d947 ,
    'returnType': 'json',
    'numOfRows': '100',
    'pageNo': '1',
    'sidoName': '서울',
    'ver': '1.0'
}

## 데이터 수집

## 호출 성공/실패 출력



SyntaxError: invalid decimal literal (3493316294.py, line 4)